# AI-Based School Safety & Real-Time Emergency Alert System

**Major project prototype using Multiclass Logistic Regression**

This notebook loads a simulated dataset, trains a three-class classifier, evaluates it, predicts a sample reading, and simulates alert generation and clearing.

> **Safety note:** This is an educational prototype. It is not connected to real sensors and must not be used for real emergency decisions.

## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## 2. Load the dataset

In [ ]:
DATA_URL = "https://github.com/YBIFoundation/Internship/raw/main/SchoolSafetyEmergencyDataset.csv"
df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the target classes

In [ ]:
class_names = {
    0: "Safe",
    1: "Normal Alert",
    2: "High-Priority Emergency"
}

print(df["severity"].value_counts().sort_index())
print("\\nClass labels:")
print(df["severity_label"].value_counts())

df["severity_label"].value_counts().plot(
    kind="bar", title="Safety Status Distribution",
    xlabel="Status", ylabel="Number of Records"
)
plt.tight_layout()
plt.show()


## 4. Select model features

`incident_type` is retained for explanation/history only; it is not used as a model input.

In [ ]:
feature_cols = [
    "temperature_c", "smoke_level", "co_ppm", "gas_level",
    "sound_db", "crowd_count", "motion_index",
    "flame_detected", "event_duration_min"
]

X = df[feature_cols]
y = df["severity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))


## 5. Build and train the Logistic Regression model

StandardScaler places numerical features on comparable scales. Balanced class weights help account for the unequal number of examples in each class.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)


## 6. Evaluate the model

In [ ]:
print(f"Accuracy: {accuracy_score(y_test, pred):.2%}")
print(classification_report(
    y_test, pred,
    labels=[0, 1, 2],
    target_names=["Safe", "Normal Alert", "High-Priority Emergency"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    labels=[0, 1, 2],
    display_labels=["Safe", "Normal Alert", "High Emergency"],
    xticks_rotation=20
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 7. Convert predictions into alert messages

In [ ]:
def create_alert_message(predicted_class):
    if predicted_class == 0:
        return "CLEAR / SAFE: No active emergency."
    elif predicted_class == 1:
        return "NORMAL ALERT: Please inspect the location."
    else:
        return "HIGH-PRIORITY EMERGENCY: Immediate response required!"

for value in [0, 1, 2]:
    print(value, "->", create_alert_message(value))


## 8. Predict a new sensor reading

In [ ]:
new_reading = pd.DataFrame([{
    "temperature_c": 65,
    "smoke_level": 88,
    "co_ppm": 40,
    "gas_level": 92,
    "sound_db": 101,
    "crowd_count": 48,
    "motion_index": 90,
    "flame_detected": 1,
    "event_duration_min": 8
}], columns=feature_cols)

prediction = int(model.predict(new_reading)[0])
probabilities = model.predict_proba(new_reading)[0]

print("Predicted status:", class_names[prediction])
print(create_alert_message(prediction))
print("\\nClass probabilities:")
for class_id, probability in zip(model.classes_, probabilities):
    print(f"{class_names[int(class_id)]:25s}: {probability:.3f}")


## 9. Simulated monitoring and automatic alert clearing

Rows from the dataset are treated as if they arrive one after another. This demonstrates the alert workflow; it is not a live sensor feed.

In [ ]:
demo_stream = pd.concat([
    df[df["severity"] == 1].sample(3, random_state=10),
    df[df["severity"] == 2].sample(4, random_state=20),
    df[df["severity"] == 0].sample(4, random_state=30),
    df.sample(9, random_state=40)
]).reset_index(drop=True)

alert_history = []
active_alert = False

for i, row in demo_stream.iterrows():
    sample = row[feature_cols].to_frame().T
    predicted = int(model.predict(sample)[0])
    predicted_label = class_names[predicted]

    if predicted == 0:
        if active_alert:
            system_action = "CLEAR: Previous alert closed automatically"
            active_alert = False
        else:
            system_action = "SAFE: No active alert"
    elif predicted == 1:
        system_action = "NORMAL ALERT: Inspection required"
        active_alert = True
    else:
        system_action = "HIGH-PRIORITY EMERGENCY: Immediate response required"
        active_alert = True

    alert_history.append({
        "stream_step": i + 1,
        "incident_type": row["incident_type"],
        "temperature_c": row["temperature_c"],
        "smoke_level": row["smoke_level"],
        "flame_detected": row["flame_detected"],
        "predicted_status": predicted_label,
        "system_action": system_action
    })

history_df = pd.DataFrame(alert_history)
display(history_df)


## 10. Latest dashboard status and export

In [ ]:
latest = history_df.iloc[-1]

print("=" * 60)
print("        SCHOOL SAFETY MONITORING DASHBOARD")
print("=" * 60)
print("Current predicted status :", latest["predicted_status"])
print("Current system action    :", latest["system_action"])
print("Temperature (C)          :", latest["temperature_c"])
print("Smoke level              :", latest["smoke_level"])
print("Flame detected           :", latest["flame_detected"])
print("=" * 60)

print("\\nRecent alert history:")
display(history_df.tail(10))

history_df.to_csv("alert_history.csv", index=False)
print("Saved alert_history.csv")


## 11. Conclusion

The prototype demonstrates a complete machine-learning workflow: simulated sensor dataset → preprocessing → multiclass model training → evaluation → prediction → alert generation → automatic clearing → alert history.

**Future enhancements:** validated IoT sensors, room-wise monitoring, a web dashboard, database storage, notification integration, acknowledgement by security staff, and human-supervised escalation rules.